In [ ]:
!git clone https://github.com/edrosten/squassh.git
import sys
import os
sys.path.insert(0, '/content/squassh')
os.environ["OVERRIDE_UNCLEAN_REPO"]="1"

In [ ]:
from typing import cast
import torch
import torch._dynamo
import resi_data         # noqa
import mark_bates_data   # noqa
import train
import network
import device
from localisation_data import LocalisationDataSetMultipleDan6

## Load in the dataset 

Load data and select some rendering parameters to give a useful rendition.

In [ ]:
nupc3d = [t.to(device.device).half() for t in resi_data.load_3d()]

SCALE=1.3
rejection = 1.0

data_parameters = train.DataParametersXYYZ(
    image_size_xy = 64,
    image_size_z = 32,
    nm_per_pixel_xy = 3*SCALE,
    z_scale = 2
)

## First training phase

Initial training starts with a small model of 35 points and decreases the rendering resolution from 65 to 34nm and then slowly to 13nm. Since there are so few points, the intensities are fixed.

In [ ]:


params_initial = train.TrainingParameters()
params_initial.batch_size = 160 
params_initial.validity_weight=rejection

params_initial.schedule[0].epochs = 90
params_initial.schedule[0].initial_psf = 50*SCALE
params_initial.schedule[0].final_psf = 26*SCALE
params_initial.schedule[0].psf_step_every= 3
params_initial.schedule[0].initial_lr= 0.0001
params_initial.schedule[0].final_lr= 0.0001

params_initial.schedule.append(train.TrainingSegment())
params_initial.schedule[1].epochs = 30
params_initial.schedule[1].initial_psf = 19*SCALE
params_initial.schedule[1].final_psf = 10.0*SCALE
params_initial.schedule[1].psf_step_every= 10
params_initial.schedule[1].initial_lr= 0.0001
params_initial.schedule[1].final_lr= 0.0001

dataset_initial = LocalisationDataSetMultipleDan6(**vars(data_parameters), data=nupc3d, augmentations=8, device=device.device)


net, parameterisation =network.PredictReconstructionStretchExpandValidDan6(model_size=35, **vars(data_parameters), data=nupc3d)
parameterisation.max_stretch_factor_axis = 2.0
parameterisation.max_stretch_factor_expand = 1.0
net.to(device.device)

net._model_intensities.requires_grad=False  # pylint: disable=protected-access

torch.compiler.reset()
fast = cast(network.GeneralPredictReconstruction, torch.compile(net))
train.retrain(fast, dataset_initial, params_initial, 'phase_0')

## Second training phase

This phase replaces each of the 35 initial points with 20 in roughly the same location, and then continues to train with a 13nm resolution with a slowly decreasing learning rate. 

In [ ]:
mult = 20
scatter = 0.01

params_refine = train.TrainingParameters()
params_refine.batch_size = 10
params_refine.validity_weight=rejection

params_refine.schedule[0].epochs = 1000
params_refine.schedule[0].initial_psf = 10.0*SCALE
params_refine.schedule[0].final_psf = 10.0*SCALE
params_refine.schedule[0].psf_step_every= 300
params_refine.schedule[0].initial_lr= 0.0002
params_refine.schedule[0].final_lr= 0.00005


scale = net.get_model()[0].abs().max().item()

old_pts, old_weights = (j.detach() for j in net.get_model())

new_pts = torch.nn.functional.interpolate(old_pts.unsqueeze(0).unsqueeze(0), scale_factor=[mult,1]).squeeze(0).squeeze(0)
new_pts += torch.randn(new_pts.shape, device=device.device) * scale * scatter

new_weights = torch.nn.functional.interpolate(old_weights.unsqueeze(0).unsqueeze(0), scale_factor=mult).squeeze(0).squeeze(0)

net.set_model(new_pts, new_weights)
net._model_intensities.requires_grad=True  # pylint: disable=protected-access
parameterisation.max_stretch_factor_expand = 1.3

torch.compiler.reset() # Otherwise it crashes on torch 2.7
fast = cast(network.GeneralPredictReconstruction, torch.compile(net))

dataset_refine = LocalisationDataSetMultipleDan6(**vars(data_parameters), data=nupc3d, augmentations=1, device=device.device)
train.retrain(fast, dataset_refine, params_refine, 'phase_1')

        

## Plot the results

Plot an XY projection, along with the axis of stretch. Note that the orientation is effectively random.


In [ ]:
import matplotlib.pyplot as plt
pts, intensities = [ i.detach().cpu() for i in net.get_model()]
s_intensities, indices = intensities.sort(descending=True) #Sort to low intensity points don't obscure high intensity ones
plt.scatter(pts[indices,0], pts[indices,1], c=s_intensities, cmap='grey')

ax = parameterisation.get_axis().cpu().detach()
ax = torch.stack([ax*30, ax*-30], 0)

plt.plot(ax[:,0], ax[:,1], 'r')



